# Federated Client Split Example

This notebook tests the split helper with the earlier example: 3 clients with custom row counts and an age-unbalanced distribution.

Equivalent CLI commands:

```bash
python3 src/split_federated_clients.py \
  --input data/default_of_credit_card_clients.xls \
  --num-clients 3 \
  --client-sizes 10000 4000 1000 \
  --unbalance-feature age \
  --unbalance-targets '[[(18, 25), 80], [(50, 70), 90], [(26, 49), 70]]' \
  --output-dir data/IID_cl3_age_unbalanced \
  --overwrite

python3 src/split_federated_clients.py \
  --input data/default_of_credit_card_clients_engineered.xlsx \
  --num-clients 3 \
  --client-sizes 10000 4000 1000 \
  --unbalance-feature age \
  --unbalance-targets '[[(18, 25), 80], [(50, 70), 90], [(26, 49), 70]]' \
  --output-dir data/IID_cl3_age_unbalanced_engineered \
  --overwrite
```

In [19]:
from pathlib import Path
import sys

import pandas as pd

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.credit_data import DEFAULT_ENGINEERED_OUTPUT_PATH, DEFAULT_INPUT_PATH, read_credit_dataset, save_engineered_dataset
from src.split_federated_clients import create_federated_clients

In [20]:
split_jobs = [
    {
        "name": "original",
        "data_path": DEFAULT_INPUT_PATH,
        "output_dir": repo_root / "data" / "IID_cl3_age_unbalanced",
    },
    {
        "name": "engineered",
        "data_path": DEFAULT_ENGINEERED_OUTPUT_PATH,
        "output_dir": repo_root / "data" / "IID_cl3_age_unbalanced_engineered",
    },
]

if not DEFAULT_ENGINEERED_OUTPUT_PATH.exists():
    save_engineered_dataset()

df = read_credit_dataset(split_jobs[0]["data_path"])

df.head()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [21]:
split_results = []

for job in split_jobs:
    data_path = Path(job["data_path"])
    output_dir = Path(job["output_dir"])
    df = read_credit_dataset(data_path)
    clients = create_federated_clients(
        df,
        num_clients=3,
        client_sizes=[1000, 3000, 1000],
        unbalance_feature="age",
        unbalance_targets=[
            ((18, 25), 80),
            ((50, 70), 60),
            ((26, 49), 70),
        ],
        output_dir=output_dir,
        source_path=data_path,
        overwrite=True,
    )
    split_results.append({"name": job["name"], "output_dir": output_dir, "clients": clients})
    print(f"Created {len(clients)} {job['name']} clients in {output_dir}")
    print("Rows:", [len(client) for client in clients])
    print("Columns per client:", len(clients[0].columns))
    print()


Created 3 clients in /homes/sguszausky/fairfed-credit-xai/data/IID_cl3_age_unbalanced
[1000, 3000, 1000]


In [22]:
for result in split_results:
    output_dir = result["output_dir"]
    if output_dir.exists():
        print(f"[{result['name']}] {output_dir}")
        print((output_dir / "show_distributions.txt").read_text())

Run this from the project root to inspect the generated client distributions:

python3 -c "from pathlib import Path; import pandas as pd; out=Path('/homes/sguszausky/fairfed-credit-xai/data/IID_cl3_age_unbalanced'); cols=['default payment next month', 'AGE']; print(pd.read_csv(out/'client_summary.csv').to_string(index=False)); print(); files=sorted(out.glob('client_*/client.csv')); [print(p.parent.name, '\\n', pd.read_csv(p)[[c for c in cols if c in pd.read_csv(p).columns]].value_counts(normalize=True).rename('share').reset_index().to_string(index=False), '\\n') for p in files]"



In [23]:
summaries = []
for result in split_results:
    output_dir = result["output_dir"]
    if output_dir.exists():
        summary = pd.read_csv(output_dir / "client_summary.csv")
        summary.insert(0, "dataset", result["name"])
        summaries.append(summary)

pd.concat(summaries, ignore_index=True)